# 20 Questions Game (Updated)
So here is an updated version of the game that features an entropy-based question selection to further reduce the number of questions

First let's import the libraries and files that I will be using. Since we are working with csv's, I will be using pandas to load the files as a dataframe. I will also be using numpy for the entropy algo.

In [2]:
import pandas as pd
import numpy as np

In [4]:
# Load data from csv files
df = pd.read_csv("data/word_attribute.csv")
qdf =pd.read_csv("data/attribute_question.csv")

Let's check to see if the data loaded properly

In [7]:
df.head()

,Index,Name,is_alive,is_big,is_animal,is_human,can_fly,can_swim,is_food,is_furniture,is_tool,is_pet,can_bark,is_clothes,is_accessory,can_write,is_liquid
0,1,Elephant,True,True,True,False,False,False,False,False,False,False,False,False,False,False,False
1,2,Chair,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False
2,3,Eagle,True,False,True,False,True,False,False,False,False,False,False,False,False,False,False
3,4,Person,True,False,True,True,False,True,False,False,False,False,False,False,False,True,False
4,5,Shark,True,True,True,False,False,True,False,False,False,False,False,False,False,False,False


In [9]:
qdf.head()

,Attributes,Questions
0,is_alive,Is it alive?
1,is_big,Is it big?
2,is_animal,Is it an animal?
3,is_human,Is it human?
4,can_fly,Can it fly?


In [11]:
# convert questions from questions csv to dictionary
questions = dict(zip(qdf["Attributes"], qdf["Questions"]))

In [13]:
# prepare object and attribute list
objects = df["Name"].tolist()
attributes = df.columns[2:]

## Functions

In [16]:
#Binary entropy function
def binary_entropy(p):
    if p in[0,1]:
        return 0
    return -p * np.log2(p) - (1 - p) * np.log2(1-p)

# Question optimization
def choose_best_question (df_subset, asked):
    best_attr = None
    max_entropy = -1

    for attr in attributes:
        if attr in asked:
            continue
        p = df_subset[attr].mean()
        entropy = binary_entropy(p)
        if entropy > max_entropy:
            max_entropy = entropy
            best_attr = attr
    return best_attr
        

## Game

In [19]:
# Start game loop
print('Think of one of these objects:')
print(', '.join(objects))
print("\nAnswer the following questions with 'y' or 'n'.") 

# initialize a dataframe that will be filtered by the given answer choices
filtered_df = df.copy()
asked = set()

while len(filtered_df) > 1 and len(asked) < len(attributes):
    attr = choose_best_question(filtered_df, asked)
    if attr is None:
        break
        
    question = questions.get(attr)
    answer = input(f"{question} (y/n): ").strip().lower()
    if answer not in ["y", "n"]:
        print("Invalid input. Please answer with 'yes' or 'no'.")
        continue

    expected = ( answer == "y")
    filtered_df = filtered_df[filtered_df[attr] == expected]
    asked.add(attr)
    

    if len(filtered_df) == 1:
        break
        
# Provide the answer        
if len(filtered_df) == 1:
    guess = filtered_df["Name"].values[0]
    print(f"\nI guess you are thinking of a: {guess}")
elif len(filtered_df) > 1:
    print("\nI'm not certain yet, but here are my best guesses:")
    print(", ".join(filtered_df["Name"].tolist()))
else:
    print("\nI couldn't guess your object. Maybe it's not in the database or the answers didn't match.")

Think of one of these objects:
Elephant, Chair, Eagle, Person, Shark, Wrench, Coin, Dog, Hat, Jeans, Cat, Pencil, House, Bee, Chicken, Cow, Bed, Sandwich, Water, Goldfish

Answer the following questions with 'y' or 'n'.


Is it alive? (y/n):  n
Is it big? (y/n):  n
Can you eat it? (y/n):  n
Is it a tool? (y/n):  n
Can you wear it? (y/n):  y
Is it used as an accessory? (y/n):  n



I guess you are thinking of a: Jeans
